## Setup

In [4]:
from pathlib import Path ## OOP way of handling file path to avoid manual slashs hanndling\
#lik ein traditonal string-based path, which the use of / or \ 
#can break different os system

from PIL import Image # to import image module from the Pillow library (Python imaging library) 

import matplotlib.pyplot as plt # for visualization


import re #regular expression to search text, and find matching pattern to clean text 

import shutil #file management function #it was imported during the file management operations like moving CSV file
## during dataset organization, ## although it was never been used in the final api call

from IPython.display import display #it was included because when I was reading real and ai image comparision
## on the other jupyter notebook 

import torch # originally this code was intended to combine with generation notebook. this is why it was imported 
import gc # originally it was imported to combine with the generation notebook 
from google.genai import types # Catching configuration mistakes immediately while typing like temperature with temp, 
# I imported because I add temperature setting when I was doing experimental setup for caption generation
# higher temperature for more uniquness # lower temperature for getting exact same descriptive caption


import os # we need this because when we loop through a spreadsheet containing 1000 images, it help to check if image exist first
import time #to track the time
import pandas as pd # to handle tabular data
from google import genai # to access Google Gemini LLMs

from google.genai.errors import APIError ## APIError was included for handling potential 
# Gemini API request failures such as rate limits, 
# connection issues, or invalid responses during automated caption generation.
from tqdm import tqdm #To wrap the processing loop 
from dotenv import load_dotenv #to load environmental variable 



import warnings
warnings.filterwarnings('ignore')

## Configuration

In [5]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Gemini Client initialized successfully!")

Gemini Client initialized successfully!


## Five Time Iteration logic

    Rationale: captions are saved by iterating the loop five times manually to retain separate files for each batch. When one csv file gets errors because of max entries exceed during API call, 404, 505 error, it will not impact on other csv files.  

## 0-999

In [6]:
## First batch

start = 0 
end = 999
input_csv = "sampled_5000_images_with_nested_labels.csv"
output_csv = f"captions_{start}_{end}.csv"

model_selection = "gemini-2.5-flash-lite"
sleep_call = 1.2 # Adds delay between API request because I got a lot of max entries exceed error without this 

def generate_shoe_caption(image_path, level_1):
    """ 
    Reusable function for opens image, sends image to Gemini, asks Gemini for a caption, fed level_1 category
    returns generated text
    """ 
    if not os.path.exists(image_path): #first image exist in the folder to prevent crashes
        return f"[ERROR: File not found: {image_path}]"
    try:
        with Image.open(image_path) as img: # to load image and close image after to save memory
            prompt = f"""
            Analyze the following shoe image. Category: **{level_1}**.
            Generate a concise text-to-image prompt (max 30 words).
            Include: "shoes facing left, entire shoes". 
            Start with the specific shoe type.
            """
            response = client.models.generate_content(model=model_selection, contents=[prompt, img]) #send image and instruction
            return response.text.strip() if response.text else "[ERROR: Empty]" # return text description
    except Exception as e: # error handling if one image fails the pipeline continue to 
        return f"[ERROR: {str(e)}]" # return error as string format

df_main = pd.read_csv(input_csv) # read dataset 
subset = df_main.iloc[start:end+1].copy()  # Extracts only current image range. 
# without this row 999 will not be included

print(f"Processing Batch: {start} to {end}...")

results = []
for index, row in tqdm(subset.iterrows(), total=len(subset)): # process image one by one in the current range
    caption = generate_shoe_caption(row['image_path'], row.get('level_1', 'footwear'))
    results.append(caption)
    time.sleep(sleep_call) # to prevent API overloading 

subset['gemini_caption'] = results 
subset.to_csv(output_csv, index=False)

print(f"Done! Saved to {output_csv}")

Processing Batch: 0 to 999...


100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [23:10<00:00,  1.39s/it]

Done! Saved to captions_0_999.csv


## 1000-1999

In [7]:
## second batch

start = 999
end = 1999
input_csv = "sampled_5000_images_with_nested_labels.csv"
output_csv = f"captions_{start}_{end}.csv"

model_selection = "gemini-2.5-flash-lite"
sleep_call = 1.2 

def generate_shoe_caption(image_path, level_1):
    if not os.path.exists(image_path):
        return f"[ERROR: File not found: {image_path}]"
    try:
        with Image.open(image_path) as img:
            prompt = f"""
            Analyze the following shoe image. Category: **{level_1}**.
            Generate a concise text-to-image prompt (max 30 words).
            Include: "shoes facing left, entire shoes". 
            Start with the specific shoe type.
            """
            response = client.models.generate_content(model=model_selection, contents=[prompt, img])
            return response.text.strip() if response.text else "[ERROR: Empty]"
    except Exception as e:
        return f"[ERROR: {str(e)}]"

df_main = pd.read_csv(input_csv)
subset = df_main.iloc[start:end+1].copy() # +1 to include the end index

print(f"Processing Batch: {start} to {end}...")

results = []
for index, row in tqdm(subset.iterrows(), total=len(subset)):
    caption = generate_shoe_caption(row['image_path'], row.get('level_1', 'footwear'))
    results.append(caption)
    time.sleep(sleep_call)

subset['gemini_caption'] = results
subset.to_csv(output_csv, index=False)

print(f"Done! Saved to {output_csv}")

Processing Batch: 999 to 1999...


100%|██████████████████████████████████████████████████████████████████████████████| 1001/1001 [22:57<00:00,  1.38s/it]

Done! Saved to captions_999_1999.csv


In [8]:
## third batch

start = 1999
end = 2999
input_csv = "sampled_5000_images_with_nested_labels.csv"
output_csv = f"captions_{start}_{end}.csv"

model_selection = "gemini-2.5-flash-lite"
sleep_call = 1.2 

def generate_shoe_caption(image_path, level_1):
    if not os.path.exists(image_path):
        return f"[ERROR: File not found: {image_path}]"
    try:
        with Image.open(image_path) as img:
            prompt = f"""
            Analyze the following shoe image. Category: **{level_1}**.
            Generate a concise text-to-image prompt (max 30 words).
            Include: "shoes facing left, entire shoes". 
            Start with the specific shoe type.
            """
            response = client.models.generate_content(model=model_selection, contents=[prompt, img])
            return response.text.strip() if response.text else "[ERROR: Empty]"
    except Exception as e:
        return f"[ERROR: {str(e)}]"

df_main = pd.read_csv(input_csv)
subset = df_main.iloc[start:end+1].copy() # +1 to include the end index

print(f"Processing Batch: {start} to {end}...")

results = []
for index, row in tqdm(subset.iterrows(), total=len(subset)):
    caption = generate_shoe_caption(row['image_path'], row.get('level_1', 'footwear'))
    results.append(caption)
    time.sleep(sleep_call)

subset['gemini_caption'] = results
subset.to_csv(output_csv, index=False)

print(f"Done! Saved to {output_csv}")

Processing Batch: 1999 to 2999...


100%|██████████████████████████████████████████████████████████████████████████████| 1001/1001 [22:53<00:00,  1.37s/it]

Done! Saved to captions_1999_2999.csv


In [9]:
## fourth batch

start = 2999
end = 3999
input_csv = "sampled_5000_images_with_nested_labels.csv"
output_csv = f"captions_{start}_{end}.csv"

model_selection = "gemini-2.5-flash-lite"
sleep_call = 1.2 

def generate_shoe_caption(image_path, level_1):
    if not os.path.exists(image_path):
        return f"[ERROR: File not found: {image_path}]"
    try:
        with Image.open(image_path) as img:
            prompt = f"""
            Analyze the following shoe image. Category: **{level_1}**.
            Generate a concise text-to-image prompt (max 30 words).
            Include: "shoes facing left, entire shoes". 
            Start with the specific shoe type.
            """
            response = client.models.generate_content(model=model_selection, contents=[prompt, img])
            return response.text.strip() if response.text else "[ERROR: Empty]"
    except Exception as e:
        return f"[ERROR: {str(e)}]"

df_main = pd.read_csv(input_csv)
subset = df_main.iloc[start:end+1].copy() # +1 to include the end index

print(f"Processing Batch: {start} to {end}...")

results = []
for index, row in tqdm(subset.iterrows(), total=len(subset)):
    caption = generate_shoe_caption(row['image_path'], row.get('level_1', 'footwear'))
    results.append(caption)
    time.sleep(sleep_call)

subset['gemini_caption'] = results
subset.to_csv(output_csv, index=False)

print(f"Done! Saved to {output_csv}")

Processing Batch: 2999 to 3999...


100%|██████████████████████████████████████████████████████████████████████████████| 1001/1001 [22:45<00:00,  1.36s/it]

Done! Saved to captions_2999_3999.csv


## Fifth batch

In [10]:
## fifth batch

start = 3999
end = 4999
input_csv = "sampled_5000_images_with_nested_labels.csv"
output_csv = f"captions_{start}_{end}.csv"

model_selection = "gemini-2.5-flash-lite"
sleep_call = 1.2 

def generate_shoe_caption(image_path, level_1):
    if not os.path.exists(image_path):
        return f"[ERROR: File not found: {image_path}]"
    try:
        with Image.open(image_path) as img:
            prompt = f"""
            Analyze the following shoe image. Category: **{level_1}**.
            Generate a concise text-to-image prompt (max 30 words).
            Include: "shoes facing left, entire shoes". 
            Start with the specific shoe type.
            """
            response = client.models.generate_content(model=model_selection, contents=[prompt, img])
            return response.text.strip() if response.text else "[ERROR: Empty]"
    except Exception as e:
        return f"[ERROR: {str(e)}]"

df_main = pd.read_csv(input_csv)
subset = df_main.iloc[start:end+1].copy() # +1 to include the end index

print(f"Processing Batch: {start} to {end}...")

results = []
for index, row in tqdm(subset.iterrows(), total=len(subset)):
    caption = generate_shoe_caption(row['image_path'], row.get('level_1', 'footwear'))
    results.append(caption)
    time.sleep(sleep_call)

subset['gemini_caption'] = results
subset.to_csv(output_csv, index=False)

print(f"Done! Saved to {output_csv}")

Processing Batch: 3999 to 4999...


100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [22:51<00:00,  1.37s/it]


Done! Saved to captions_3999_4999.csv
